# Feature Analysis: Acoustic Features for Dysarthria Detection

This notebook analyzes extracted acoustic features:
- MFCC distributions and patterns
- Prosodic features (F0, energy, speaking rate)
- Formant features (vowel space)
- Feature importance and separability

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import librosa
from scipy import stats

from src.features.mfcc_extractor import MFCCExtractor
from src.features.prosodic_extractor import ProsodicExtractor
from src.features.formant_extractor import FormantExtractor

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Extract Features from Sample Audio

In [ ]:
# Load manifests
train_df = pd.read_csv('../data/manifests/train.csv')

# Sample files
healthy_files = train_df[train_df['label'] == 0].sample(n=20)['file_path'].tolist()
dysarthric_files = train_df[train_df['label'] == 1].sample(n=20)['file_path'].tolist()

print(f"Analyzing {len(healthy_files)} healthy and {len(dysarthric_files)} dysarthric samples...")

In [ ]:
# Initialize extractors
mfcc_extractor = MFCCExtractor()
prosody_extractor = ProsodicExtractor()
formant_extractor = FormantExtractor()

def extract_features(file_path):
    """Extract all features from audio file."""
    y, sr = librosa.load(file_path, sr=16000)
    
    mfcc_features = mfcc_extractor.extract(y, sr)
    prosody_features = prosody_extractor.extract(y, sr)
    formant_features = formant_extractor.extract(y, sr)
    
    return {
        'mfcc_mean': mfcc_features.mfcc.mean(),
        'mfcc_std': mfcc_features.mfcc.std(),
        'f0_mean': prosody_features.f0_mean,
        'f0_std': prosody_features.f0_std,
        'energy_mean': prosody_features.energy_mean,
        'speaking_rate': prosody_features.speaking_rate,
        'pause_ratio': prosody_features.pause_ratio,
        'f1_mean': formant_features.f1_mean,
        'f2_mean': formant_features.f2_mean,
        'vowel_space_area': formant_features.vowel_space_area,
    }

# Extract features
healthy_features = [extract_features(f) for f in healthy_files]
dysarthric_features = [extract_features(f) for f in dysarthric_files]

healthy_df = pd.DataFrame(healthy_features)
healthy_df['label'] = 0

dysarthric_df = pd.DataFrame(dysarthric_features)
dysarthric_df['label'] = 1

features_df = pd.concat([healthy_df, dysarthric_df], ignore_index=True)
print("Feature extraction complete!")

## 2. Prosodic Feature Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# F0 mean
axes[0, 0].boxplot([healthy_df['f0_mean'], dysarthric_df['f0_mean']], labels=['Healthy', 'Dysarthric'])
axes[0, 0].set_ylabel('F0 Mean (Hz)', fontsize=12)
axes[0, 0].set_title('Fundamental Frequency', fontsize=14, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# Energy
axes[0, 1].boxplot([healthy_df['energy_mean'], dysarthric_df['energy_mean']], labels=['Healthy', 'Dysarthric'])
axes[0, 1].set_ylabel('Energy Mean', fontsize=12)
axes[0, 1].set_title('Energy', fontsize=14, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Speaking rate
axes[1, 0].boxplot([healthy_df['speaking_rate'], dysarthric_df['speaking_rate']], labels=['Healthy', 'Dysarthric'])
axes[1, 0].set_ylabel('Speaking Rate (syllables/s)', fontsize=12)
axes[1, 0].set_title('Speaking Rate', fontsize=14, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# Pause ratio
axes[1, 1].boxplot([healthy_df['pause_ratio'], dysarthric_df['pause_ratio']], labels=['Healthy', 'Dysarthric'])
axes[1, 1].set_ylabel('Pause Ratio', fontsize=12)
axes[1, 1].set_title('Pause Ratio', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Formant Feature Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1-F2 vowel space
axes[0].scatter(healthy_df['f2_mean'], healthy_df['f1_mean'], alpha=0.6, s=100, label='Healthy', color='#2ecc71')
axes[0].scatter(dysarthric_df['f2_mean'], dysarthric_df['f1_mean'], alpha=0.6, s=100, label='Dysarthric', color='#e74c3c')
axes[0].set_xlabel('F2 (Hz)', fontsize=12)
axes[0].set_ylabel('F1 (Hz)', fontsize=12)
axes[0].set_title('Vowel Space (F1 vs F2)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].grid(alpha=0.3)
axes[0].invert_xaxis()  # Conventional F1-F2 plot
axes[0].invert_yaxis()

# Vowel space area
axes[1].boxplot([healthy_df['vowel_space_area'], dysarthric_df['vowel_space_area']], labels=['Healthy', 'Dysarthric'])
axes[1].set_ylabel('Vowel Space Area (Hz²)', fontsize=12)
axes[1].set_title('Vowel Space Area', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Statistical Significance Tests

In [ ]:
# T-tests for feature separability
feature_cols = ['f0_mean', 'energy_mean', 'speaking_rate', 'pause_ratio', 'vowel_space_area']

results = []
for feature in feature_cols:
    healthy_vals = healthy_df[feature].dropna()
    dysarthric_vals = dysarthric_df[feature].dropna()
    
    t_stat, p_value = stats.ttest_ind(healthy_vals, dysarthric_vals)
    
    results.append({
        'Feature': feature,
        'Healthy Mean': healthy_vals.mean(),
        'Dysarthric Mean': dysarthric_vals.mean(),
        'Difference': abs(healthy_vals.mean() - dysarthric_vals.mean()),
        't-statistic': t_stat,
        'p-value': p_value,
        'Significant': 'Yes' if p_value < 0.05 else 'No',
    })

results_df = pd.DataFrame(results)
results_df.sort_values('p-value')

## 5. Feature Correlation

In [ ]:
# Correlation matrix
corr_matrix = features_df[feature_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

Key observations:
- **F0 (pitch)**: Dysarthric speech may show different pitch patterns
- **Speaking rate**: Dysarthric speech often slower
- **Pause ratio**: More pauses in dysarthric speech
- **Vowel space**: Reduced vowel space area indicates articulatory impairment